<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/04_polynomial_ridge_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S4 多項式回帰・リッジ回帰と過学習(notebook)

本文:[`04-polynomial-ridge.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/supervised/04-polynomial-ridge.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`supervised/04_polynomial_ridge.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

def make_data(n, noise=0.25, seed=0):
    # 本当の関係 sin(2πx) に、標準偏差 noise の雑音を足したデータ
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    t = np.sin(2 * np.pi * x) + rng.normal(0, noise, n)
    return x, t

x_train, t_train = make_data(10, seed=1)     # 学習用(10個)
x_test, t_test = make_data(100, seed=2)      # 評価用(100個):学習には使わない
xs = np.linspace(0, 1, 200)                  # 曲線を描くための点

## 1. 計画行列

本文 §1 の計画行列 $\Phi$(形 `(N, M+1)`、 $\Phi_{nj} = x_n^j$)を作る。`np.vander(x, M + 1, increasing=True)` が使える。

**手計算**:本文の確認課題1( $x = 0, 0.5, 1$ 、 $M = 2$)を解く。

- 手計算:(ここに書く)

In [ ]:
def design_matrix(x, M):
    return np.vander(x, M + 1, increasing=True)

In [ ]:
check("確認課題1", design_matrix(np.array([0.0, 0.5, 1.0]), 2), [[1, 0, 0], [1, 0.5, 0.25], [1, 1, 1]])
check("形は (N, M+1)", design_matrix(x_train, 9).shape, (10, 10))

## 2. 最小二乗法で当てはめる

S2 と同じ最小二乗法である。ただし、次数が大きいと $\Phi^\top \Phi$ の条件数が大きくなるので、`lstsq` を使う(本文 §2 の WARNING)。

**予想**:$M = 0, 1, 3, 9$ で、学習用の誤差と評価用の誤差はそれぞれどうなるか。

- 予想:(ここに書く)

In [ ]:
def fit_least_squares(x, t, M):
    return np.linalg.lstsq(design_matrix(x, M), t, rcond=None)[0]

In [ ]:
def rmse(x, t, w):
    M = len(w) - 1
    return np.sqrt(np.mean((design_matrix(x, M) @ w - t) ** 2))

w9 = fit_least_squares(x_train, t_train, 9)
check("M = 9 なら 10 個の点をすべて通る(学習用の誤差 0)", rmse(x_train, t_train, w9), 0.0, tol=1e-6)
check("M = 1 は直線の最小二乗法(np.polyfit と同じ)", fit_least_squares(x_train, t_train, 1), np.polyfit(x_train, t_train, 1)[::-1])

print(" M | 学習用 RMSE | 評価用 RMSE | 係数の絶対値の最大 | Phi^T Phi の条件数")
for M in [0, 1, 3, 6, 9]:
    w = fit_least_squares(x_train, t_train, M)
    Phi = design_matrix(x_train, M)
    print(f"{M:2d} | {rmse(x_train, t_train, w):11.3f} | {rmse(x_test, t_test, w):11.3f} | {np.abs(w).max():18.1f} | {np.linalg.cond(Phi.T @ Phi):.1e}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, M in zip(axes, [0, 1, 3, 9]):
    w = fit_least_squares(x_train, t_train, M)
    ax.scatter(x_train, t_train, c="b", s=20, label="train")
    ax.plot(xs, np.sin(2 * np.pi * xs), "g--", label="true")
    ax.plot(xs, design_matrix(xs, M) @ w, "r", label=f"M={M}")
    ax.set_ylim(-1.8, 1.8); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. リッジ回帰

本文の式 (2):$(\Phi^\top \Phi + \lambda I) w = \Phi^\top t$ を `np.linalg.solve` で解く。 $\lambda > 0$ なら条件数が小さくなり、`solve` で安定に解ける。

**予想**:$M = 9$ のまま $\lambda$ を 0 から大きくしていくと、評価用の誤差はどう変わるか。

- 予想:(ここに書く)

In [ ]:
def fit_ridge(x, t, M, lam):
    Phi = design_matrix(x, M)
    return np.linalg.solve(Phi.T @ Phi + lam * np.eye(M + 1), Phi.T @ t)

In [ ]:
check("lam がとても小さいと、最小二乗法(M=3)に近い", fit_ridge(x_train, t_train, 3, 1e-12), fit_least_squares(x_train, t_train, 3), tol=1e-5)
check("lam がとても大きいと、係数はほぼ 0", fit_ridge(x_train, t_train, 9, 1e8), np.zeros(10), tol=1e-3)

lams = [1e-8, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
tr = [rmse(x_train, t_train, fit_ridge(x_train, t_train, 9, l)) for l in lams]
te = [rmse(x_test, t_test, fit_ridge(x_train, t_train, 9, l)) for l in lams]
plt.semilogx(lams, tr, "o-", label="train"); plt.semilogx(lams, te, "o-", label="test")
plt.xlabel("lambda"); plt.ylabel("RMSE"); plt.title("ridge, M = 9"); plt.legend(); plt.grid(); plt.show()
for l, a, b in zip(lams, tr, te):
    print(f"lambda = {l:7.0e}: 学習用 {a:.3f}  評価用 {b:.3f}  係数の絶対値の最大 {np.abs(fit_ridge(x_train, t_train, 9, l)).max():8.2f}")

## 4. 交差検証で $\lambda$ を選ぶ

本文 §4 の $K$ 分割交差検証を実装する。評価用のデータ(`x_test`)は**使わない**。ここでは、学習用のデータを 50 個に増やして使う。

**実装**:`cv_rmse` の `TODO` を書く。`folds[i]` を検証用、残りを学習用にする。

In [ ]:
def cv_rmse(x, t, M, lam, K=5, seed=0):
    idx = np.random.default_rng(seed).permutation(len(x))
    folds = np.array_split(idx, K)
    errors = []
    for i in range(K):
        val = folds[i]
        trn = np.concatenate([folds[j] for j in range(K) if j != i])
        w = fit_ridge(x[trn], t[trn], M, lam)
        errors.append(rmse(x[val], t[val], w))
    return np.mean(errors)

In [ ]:
x50, t50 = make_data(50, seed=3)
scores = {l: cv_rmse(x50, t50, 9, l) for l in lams}
best = min(scores, key=scores.get)
for l, s in scores.items():
    print(f"lambda = {l:7.0e}: 交差検証の RMSE {s:.3f}" + ("  ← 最小" if l == best else ""))
w_best = fit_ridge(x50, t50, 9, best)                       # 選んだ lambda で、学習用のデータ全体を使って学習し直す
print(f"選んだ lambda = {best:.0e} で、最後に1回だけテスト用のデータで測る:RMSE {rmse(x_test, t_test, w_best):.3f}")
check("交差検証の誤差は、データを分ける seed によらず同じ形(最小の位置がほぼ同じ)", abs(np.log10(min(lams, key=lambda l: cv_rmse(x50, t50, 9, l, seed=1))) - np.log10(best)) <= 1, True)

## 5. バイアスとバリアンスを見る

同じ作り方で学習用のデータ(10 個)を 30 組作り、それぞれに当てはめた曲線を重ねて描く(本文 §5)。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (M, lam, title) in zip(axes, [(1, 0.0, "M=1 (simple)"), (9, 1e-8, "M=9, lambda=1e-8 (complex)"), (9, 1e-3, "M=9, lambda=1e-3")]):
    curves = []
    for k in range(30):
        x_k, t_k = make_data(10, seed=100 + k)
        w = fit_ridge(x_k, t_k, M, lam) if lam > 0 else fit_least_squares(x_k, t_k, M)
        curves.append(design_matrix(xs, M) @ w)
        ax.plot(xs, curves[-1], "r", alpha=0.15)
    curves = np.array(curves)
    ax.plot(xs, curves.mean(axis=0), "k", label="mean of 30 fits")
    ax.plot(xs, np.sin(2 * np.pi * xs), "g--", label="true")
    bias2 = np.mean((curves.mean(axis=0) - np.sin(2 * np.pi * xs)) ** 2)
    var = np.mean(curves.var(axis=0))
    ax.set_title(f"{title}\nbias^2={bias2:.3f}, variance={var:.3f}", fontsize=9)
    ax.set_ylim(-2, 2); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 6. 条件を変える

1. 学習用のデータを 10 個から 100 個に増やすと、 $M = 9$ (正則化なし)の評価用の誤差はどうなるか
2. 雑音の大きさ `noise` を 0.05 と 0.5 にすると、交差検証で選ばれる $\lambda$ はどう変わるか

In [ ]:
for n in [10, 30, 100]:
    xn, tn = make_data(n, seed=1)
    print(f"1. 学習用 {n:3d} 個:M = 9 の評価用 RMSE {rmse(x_test, t_test, fit_least_squares(xn, tn, 9)):.3f}")
for noise in [0.05, 0.25, 0.5]:
    xn, tn = make_data(50, noise=noise, seed=3)
    sc = {l: cv_rmse(xn, tn, 9, l) for l in lams}
    print(f"2. noise = {noise}:交差検証で選ばれた lambda = {min(sc, key=sc.get):.0e}")

## 7. 説明(自分の言葉で書く)

1. 次数を上げると、学習用の誤差が下がるのに評価用の誤差が上がるのはなぜか
   - (ここに書く)
2. リッジ回帰の $\lambda$ は何をしているか。大きすぎるとどうなるか
   - (ここに書く)
3. ハイパーパラメータを選ぶのに、テスト用のデータを使ってはいけないのはなぜか
   - (ここに書く)